In [4]:
import numpy as np
import qutip as qt
from qitf_model import *
from tqdm import tqdm
import pickle
from entropy import RDM_entropy
import random as rand
import scipy.linalg as sc
N=10
d=2**N

exact_model = QITFModel(hx=0.809,J=1)
# analog_model = analog_QITF(exact_model, delta=[(rand.normalvariate(0, 0.001)) for _ in range(10)],eta=0.001)
analog_model = analog_QITF(exact_model, delta=0,eta=0.001)


def thermal_state(temperature):
    computation_basis =[[0,0,0,0],[0,1,0,1],[1,0,1,0],[1,1,1,1]]
    H=qt.Qobj(np.zeros((d,d)), dims=[[2]*10]*2)
    for i in range(4):
        H-=qt.tensor([qt.sigmaz() if j == i else qt.qeye(2) for j in range(10)])
    # psi_T = qt.Qobj(np.zeros(d), dims=[2]*10)
    psi_T = qt.Qobj(np.zeros((d,1)), dims=[[2]*10, [1]])
    for A_subsys in computation_basis:
        psi_T += qt.tensor(qt.basis([2]*4,A_subsys),qt.basis([2]*6))
    # Calculate the thermal state using the Gibbs distribution
    psi_T = (-H * temperature).expm()*psi_T
    # Normalize the density matrix
    return psi_T.unit()

def analog_error(initial_state, analog_model : analog_QITF,time):
    # Compute the error between the initial state and the analog model
    exact_states=sc.expm(-1j*analog_model.exact_model.H*time)@initial_state
    analog_states=sc.expm(-1j*analog_model.H*time)@initial_state
    error_norms = np.linalg.norm(exact_states-analog_states)
    return error_norms

seed=123

entropy_list = []
error_list = []
# initial_state = qt.rand_ket([2]*10,seed = 123)
# initial_state = qt.basis([2]*10)
for temperature in tqdm(range(60)):
    state = thermal_state((temperature)*0.008).full()
    entropy_list.append(RDM_entropy(state,2)[-1]) #entropy of 2-qubit subsystem
    error_list.append(analog_error(state,analog_model,1))

100%|██████████| 60/60 [30:59<00:00, 30.99s/it]


In [5]:
import pickle
with open('QIMF_EvE_entropy.pkl', 'wb') as f:
    pickle.dump((error_list, entropy_list), f)